# CelebA STAQ Debug

A lightweight notebook for fast local debugging of the CelebA STAQ path.

Use this notebook for:
- tiny CPU-only sanity runs
- first-query probe checks
- quick replay inspection
- per-concept Concept-QA spot checks

This is intentionally not the canonical experiment notebook.

In [ ]:
%pip install -e ..

In [58]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from staq.analysis import (
    evaluate_bundles_on_fixed_histories,
    plot_rollout_comparisons,
    probe_topk_sensitive_queries,
    sample_intuition_replays,
)
from staq.config import CelebAStaqConfig, default_paths
from staq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from staq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    get_celeba_loaders,
    get_raw_celeba_dataset,
    load_celeba_attribute_spec,
)
from staq.models import ConceptNet2
from staq.training import HistorySamplingConfig, build_staq_models, fit_concept_qa, fit_staq, seed_everything

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = CelebAStaqConfig()
device = torch.device("cpu")
seed_everything(config.random_seed)

figure_ext = ".svg"
download_celeba = False

# Reuse a full-run QA checkpoint if you already have one.
qa_source_experiment = "celeba_attractive"
experiment_name = "celeba_debug_attractive_lambda_gender"

debug_force_retrain = False
debug_epochs = 3
debug_batch_size = 16
debug_num_workers = 0

concept_qa_max_train_batches = 10
concept_qa_max_eval_batches = 3
staq_max_train_batches = 30
staq_max_eval_batches = 10

train_min_history = 0
train_max_history = 8
train_non_sensitive_only = False
sensitive_target_mode = "max"
sample_sensitive_attribute = "Male"
actor_eps_end = 0.2
actor_eps_anneal_epochs = debug_epochs
lambda_values = [0.0, 0.2, 0.4, 0.8]
lambda_seeds = {lambda_adv: config.random_seed + idx for idx, lambda_adv in enumerate(lambda_values)}

rollout_min_history = 0
rollout_max_history = 0
rollout_history_mode = "random"
replay_max_steps = 3

probe_pool_size = 600
comparison_max_samples = 600
comparison_num_trials = 3


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")
print(f"experiment_name: {experiment_name}")
print(f"qa_source_experiment: {qa_source_experiment}")
print(f"sample sensitive attribute: {sample_sensitive_attribute}")
print(f"sensitive_target_mode: {sensitive_target_mode}")
print(f"lambda values: {lambda_values}")
print(f"lambda seeds: {lambda_seeds}")
print(f"debug epochs: {debug_epochs}")
print(f"batch size: {debug_batch_size}")
print(f"STAQ batch caps: train={staq_max_train_batches}, eval={staq_max_eval_batches}")

repo_root: /Users/amir.atashin/Projects/conceptqa_vip-main
artifacts_root: /Users/amir.atashin/Projects/conceptqa_vip-main/artifacts
device: cpu
experiment_name: celeba_debug_lambda_gender_medium_v1
qa_source_experiment: celeba_smiling_filtered_shortcuts_rs16_anneal_lam08_widesens
sample sensitive attribute: Male
sensitive_target_mode: max
lambda values: [0.0, 0.2, 0.4, 0.8]
lambda seeds: {0.0: 0, 0.2: 1, 0.4: 2, 0.8: 3}
debug epochs: 3
batch size: 16
STAQ batch caps: train=30, eval=10


In [60]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    download=download_celeba,
)
positive_class_idx = 1
positive_class_name = spec.class_names[positive_class_idx]
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)
sample_sens_idx = torch.tensor([spec.query_attribute_names.index(sample_sensitive_attribute)], dtype=torch.long)

qa_train_loader, qa_valid_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=debug_batch_size,
    num_workers=debug_num_workers,
    download=download_celeba,
)
staq_train_loader, staq_valid_loader, _ = get_celeba_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=debug_batch_size,
    num_workers=debug_num_workers,
    return_query_targets=True,
    download=download_celeba,
)
_, _, test_ds = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=False,
    download=download_celeba,
)
raw_test_ds = get_raw_celeba_dataset(paths.data_root, spec=spec, split="test", download=False)

print(f"target attribute: {spec.target_attribute}")
print(f"# queries: {len(concepts)}")
print(f"sensitive query attributes: {spec.sensitive_attribute_names}")
print(f"sample sensitive target: {sample_sensitive_attribute}")
print(f"train/valid/test sizes: {len(staq_train_loader.dataset)}, {len(staq_valid_loader.dataset)}, {len(test_ds)}")

target attribute: Smiling
# queries: 36
sensitive query attributes: ['Male', 'No_Beard', 'Mustache', 'Goatee', 'Sideburns', '5_o_Clock_Shadow', 'Heavy_Makeup', 'Wearing_Lipstick']
sample sensitive target: Male
train/valid/test sizes: 162770, 19867, 19962


In [61]:
all_targets = staq_train_loader.dataset.base_dataset.attr.float()
all_attr_names = list(staq_train_loader.dataset.base_dataset.attr_names)
if len(all_attr_names) != all_targets.size(1):
    all_attr_names = [name for name in all_attr_names if name and name.strip()]

attr_to_idx = {name: idx for idx, name in enumerate(all_attr_names)}
gender_values = all_targets[:, attr_to_idx["Male"]]

def _phi_binary(x, y):
    x = x.bool()
    y = y.bool()
    n11 = float((x & y).sum().item())
    n10 = float((x & ~y).sum().item())
    n01 = float((~x & y).sum().item())
    n00 = float((~x & ~y).sum().item())
    denom = ((n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)) ** 0.5
    return 0.0 if denom == 0 else (n11 * n00 - n10 * n01) / denom

attribute_stats = []
for attr_name in all_attr_names:
    values = all_targets[:, attr_to_idx[attr_name]]
    gender_yes = gender_values == 1
    gender_no = gender_values == 0
    attribute_stats.append(
        {
            "attribute": attr_name,
            "rate": round(float(values.mean().item()), 4),
            "p_attr_given_gender_1": round(float(values[gender_yes].mean().item()), 4),
            "p_attr_given_gender_0": round(float(values[gender_no].mean().item()), 4),
            "phi_with_gender": round(float(_phi_binary(values, gender_values)), 4),
            "is_sensitive_query": attr_name in spec.sensitive_attribute_names,
            "is_target": attr_name == config.target_attribute,
        }
    )

attribute_stats_sorted = sorted(attribute_stats, key=lambda row: abs(row["phi_with_gender"]), reverse=True)
attribute_stats_sorted

[{'attribute': 'Male',
  'rate': 0.4194,
  'p_attr_given_gender_1': 1.0,
  'p_attr_given_gender_0': 0.0,
  'phi_with_gender': 1.0,
  'is_sensitive_query': True,
  'is_excluded_query': False,
  'is_target': False},
 {'attribute': 'Wearing_Lipstick',
  'rate': 0.4696,
  'p_attr_given_gender_1': 0.0065,
  'p_attr_given_gender_0': 0.8041,
  'phi_with_gender': -0.7886,
  'is_sensitive_query': True,
  'is_excluded_query': False,
  'is_target': False},
 {'attribute': 'Heavy_Makeup',
  'rate': 0.3843,
  'p_attr_given_gender_1': 0.003,
  'p_attr_given_gender_0': 0.6597,
  'phi_with_gender': -0.6663,
  'is_sensitive_query': True,
  'is_excluded_query': False,
  'is_target': False},
 {'attribute': 'No_Beard',
  'rate': 0.8342,
  'p_attr_given_gender_1': 0.6063,
  'p_attr_given_gender_0': 0.9988,
  'phi_with_gender': -0.5207,
  'is_sensitive_query': True,
  'is_excluded_query': False,
  'is_target': False},
 {'attribute': '5_o_Clock_Shadow',
  'rate': 0.1117,
  'p_attr_given_gender_1': 0.266,
  'p

In [70]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

attribute_stats_df = pd.DataFrame(attribute_stats_sorted)
attribute_stats_df.sort_values("rate", ascending=False, inplace=True)
attribute_stats_df

,attribute,rate,p_attr_given_gender_1,p_attr_given_gender_0,phi_with_gender,is_sensitive_query,is_excluded_query,is_target
3,No_Beard,0.8342,0.6063,0.9988,-0.5207,True,False,False
14,Young,0.7789,0.6365,0.8818,-0.2918,False,False,False
6,Attractive,0.5136,0.2785,0.6834,-0.3997,False,False,False
35,Mouth_Slightly_Open,0.4822,0.4236,0.5245,-0.0997,False,True,False
29,Smiling,0.4797,0.3993,0.5377,-0.1367,False,False,True
1,Wearing_Lipstick,0.4696,0.0065,0.8041,-0.7886,True,False,False
17,High_Cheekbones,0.4524,0.3072,0.5573,-0.2480,False,True,False
0,Male,0.4194,1.0000,0.0000,1.0000,True,False,False
2,Heavy_Makeup,0.3843,0.0030,0.6597,-0.6663,True,False,False
10,Wavy_Hair,0.3194,0.1430,0.4467,-0.3215,False,False,False


In [ ]:
source_qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_source_experiment}.pt"

if source_qa_checkpoint.exists() and not debug_force_retrain:
    answering_model = load_concept_qa_checkpoint(source_qa_checkpoint, device=device)
    qa_source = source_qa_checkpoint
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        qa_optimizer,
        T_max=max(debug_epochs, 1),
    )
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_valid_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=debug_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=None,
        clip_device=device,
        train_device=device,
        max_train_batches=concept_qa_max_train_batches,
        max_eval_batches=concept_qa_max_eval_batches,
    )
    answering_model = qa_model.eval()
    qa_source = "in-memory debug Concept-QA"

print(f"Concept-QA ready from: {qa_source}")

In [ ]:
def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )


def lambda_run_name(lambda_adv):
    return "baseline" if lambda_adv == 0 else f"lam_{lambda_adv:.2f}"


def load_or_train_bundle(run_name, lambda_adv, run_seed):
    seed_everything(run_seed)
    actor, classifier, s_head = build_staq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=config.learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=train_min_history,
        max_history=train_max_history,
        non_sensitive_only=train_non_sensitive_only,
    )
    history, best = fit_staq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=staq_train_loader,
        test_loader=staq_valid_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_adv=lambda_adv,
        alpha_sens=0.0,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=debug_epochs,
        max_train_batches=staq_max_train_batches,
        max_test_batches=staq_max_eval_batches,
        actor_eps_end=actor_eps_end,
        actor_eps_anneal_epochs=actor_eps_anneal_epochs,
        sensitive_target_mode=sensitive_target_mode,
        sensitive_target_indices=sample_sens_idx,
    )
    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    actor.eval()
    classifier.eval()
    return {
        "ckpt_path": "in-memory debug run",
        "meta": {
            "run_name": run_name,
            "lambda_adv": lambda_adv,
            "run_seed": run_seed,
            "sample_sensitive_attribute": sample_sensitive_attribute,
            "query_sensitive_attributes": spec.sensitive_attribute_names,
            "sensitive_target_mode": sensitive_target_mode,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
        },
        "actor": actor,
        "classifier": classifier,
        "history": history,
    }


bundles = {
    lambda_run_name(lambda_adv): load_or_train_bundle(
        lambda_run_name(lambda_adv),
        lambda_adv=lambda_adv,
        run_seed=lambda_seeds[lambda_adv],
    )
    for lambda_adv in lambda_values
}
baseline_bundle = bundles["baseline"]
staq_bundle = bundles[lambda_run_name(max(lambda_values))]

for name, bundle in bundles.items():
    print(name, "->", bundle["ckpt_path"])


In [ ]:
history_summaries = []
for lambda_adv in lambda_values:
    name = lambda_run_name(lambda_adv)
    final = bundles[name]["history"][-1]
    history_summaries.append(
        {
            "run_name": name,
            "lambda_adv": lambda_adv,
            "train_acc": round(final["train_acc"], 4),
            "test_acc": round(final["test_acc"], 4),
            "train_sens_acc": round(final["train_sens_acc"], 4),
            "test_sens_acc": round(final["test_sens_acc"], 4),
            "train_sensitive_query_rate": round(final["train_sens_q_rate"], 4),
            "test_sensitive_query_rate": round(final["test_sens_q_rate"], 4),
            "actor_eps": round(final["actor_eps"], 4),
        }
    )

history_summaries

In [ ]:
selected_concepts = [
    "Oval_Face",
    "Bags_Under_Eyes",
    "Wearing_Earrings",
    "Male",
    "No_Beard",
    "Heavy_Makeup",
    "Wearing_Lipstick",
]

selected_indices = [spec.query_attribute_names.index(name) for name in selected_concepts if name in spec.query_attribute_names]
selected_labels = [spec.query_attribute_names[idx] for idx in selected_indices]

qa_rows = []
seen_batches = 0
for images, _, concept_targets in staq_valid_loader:
    preds = (answer_builder(images) > 0).float()
    targets = concept_targets.float()
    for idx, attr_name in zip(selected_indices, selected_labels):
        qa_rows.append(
            {
                "attribute": attr_name,
                "accuracy": float((preds[:, idx] == targets[:, idx]).float().mean().item()),
                "pred_positive_rate": float(preds[:, idx].mean().item()),
                "target_positive_rate": float(targets[:, idx].mean().item()),
            }
        )
    seen_batches += 1
    if seen_batches >= 3:
        break

qa_rows

In [ ]:
first_query_probes = {}
for name, bundle in bundles.items():
    first_query_probes[name] = probe_topk_sensitive_queries(
        dataset=test_ds,
        answer_builder=answer_builder,
        bundle=bundle,
        concepts=concepts,
        sensitive_mask=sensitive_mask,
        class_names=spec.class_names,
        matched_sensitive_concepts=spec.sensitive_attribute_names,
        pool_size=probe_pool_size,
        num_trials=2,
        min_history=rollout_min_history,
        max_history=rollout_max_history,
        history_mode=rollout_history_mode,
        topk_steps=1,
        random_seed=config.random_seed,
        label_tag="S",
    )

{
    name: {
        "top_first_queries": probe["summary"]["top_overall_concepts_in_topk"][:8],
        "top_sensitive_first_queries": probe["summary"]["top_sensitive_concepts"][:8],
        "sensitive_in_topk_rate": probe["summary"]["sensitive_in_topk_rate"],
    }
    for name, probe in first_query_probes.items()
}

In [ ]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    num_cases=2,
    pool_size=200,
    num_trials=2,
    min_history=rollout_min_history,
    max_history=rollout_max_history,
    history_mode=rollout_history_mode,
    prefer_baseline_sensitive=False,
    confidence_threshold=config.confidence_threshold,
    rollout_max_steps=replay_max_steps,
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path(f"{experiment_name}_replay_examples"),
    title_prefix="debug replay",
)

print(f"Saved debug replay figure: {intuition_fig}")

In [ ]:
comparison_rows = evaluate_bundles_on_fixed_histories(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundles_by_name=bundles,
    sensitive_mask=sensitive_mask,
    min_history=1,
    max_history=2,
    history_mode="non_sensitive",
    num_trials=comparison_num_trials,
    max_samples=comparison_max_samples,
    eval_seed=config.random_seed,
    desc="Debug fixed-history comparison",
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
)

[
    {
        "run_name": row["run_name"],
        "lambda_adv": row["lambda_adv"],
        "mean_acc": round(row["mean_acc"], 4),
        "mean_sensitive_query_rate": round(row["mean_sensitive_query_rate"], 4),
        "mean_confidence": round(row["mean_confidence"], 4),
        "mean_positive_prob": round(row["mean_positive_prob"], 4),
    }
    for row in comparison_rows
]